In [1]:
import numpy as np
import pandas as pd

In [2]:
path = r"C:\Users\JOKER\Desktop\berlin\Fächer\Summer School\Power Markets\Data"
load = pd.read_csv(f"{path}/forecast_load.csv")  #made from task1
da = pd.read_csv(f"{path}/day_ahead_prices.csv")

In [3]:
grid = load[["timestamp_utc", "total_forecast_mwh"]].merge(da[["timestamp_utc", "day_ahead_price_eur_mwh"]], on="timestamp_utc")
L = grid["total_forecast_mwh"].to_numpy()
P = grid["day_ahead_price_eur_mwh"].to_numpy()

In [4]:
grid

,timestamp_utc,total_forecast_mwh,day_ahead_price_eur_mwh
0,2023-12-31T23:00:00Z,1.077136,0.10
1,2023-12-31T23:15:00Z,1.003301,0.10
2,2023-12-31T23:30:00Z,0.933171,0.10
3,2023-12-31T23:45:00Z,0.865711,0.10
4,2024-01-01T00:00:00Z,0.800915,0.01
...,...,...,...
35131,2024-12-31T21:45:00Z,1.454299,9.06
35132,2024-12-31T22:00:00Z,1.395945,0.52
35133,2024-12-31T22:15:00Z,1.324609,0.52
35134,2024-12-31T22:30:00Z,1.246778,0.52


In [ ]:
def run(name, tag):
    # UNHEDGED has no futures at all, so the hedge profile and its cost are zero
    if tag is None:
        H = np.zeros(len(grid))
        c_futures = 0.0
    else:
        H = pd.read_csv(f"{path}/task3_hedge_" + tag + ".csv")["hedge_energy"].to_numpy()
        pos = pd.read_csv(f"{path}/task3_positions_" + tag + ".csv")
        c_futures = (pos["hedge_energy"] * pos["price_eur_mwh"]).sum()

    # Residual volume left to buy or sell Day-Ahead
    residual = L - H

    # Physical representation: pay for the futures, then settle the residual Day-Ahead
    c_forecast = c_futures + residual @ P

    # Financial representation: buy everything Day-Ahead, then add the futures profit
    profit_futures = H @ P - c_futures
    c_financial = L @ P - profit_futures
    print("\n", name)
    print("  futures cost:", round(c_futures, 2), "EUR")
    print("  futures profit:", round(profit_futures, 2), "EUR")
    print("  C_forecast physical:", round(c_forecast, 2), "EUR")  #hedge cost + spot cost
    print("  C_forecast financial:", round(c_financial, 2), "EUR")  #spot cost - hedge profit
    print("  reconciliation difference:", round(c_forecast - c_financial, 6), "EUR")  #checking the last 2
    print("  positive residual (to buy):", round(residual[residual > 0].sum(), 2), "MWh")  #shortage energy
    print("  negative residual (to sell):", round(residual[residual < 0].sum(), 2), "MWh")  #surplus energy

    pd.DataFrame({"timestamp_utc": grid["timestamp_utc"], "hedge_energy": H, "day_ahead_residual": residual}).to_csv(f"{path}/residual_" + str(tag) + ".csv", index=False)

In [6]:
run("UNHEDGED", None)
run("Coarse hedge (CAL)", "coarse")
run("Granular hedge (M01-M03, Q2-Q4)", "granular")


 UNHEDGED
  futures cost: 0.0 EUR
  futures profit: 0.0 EUR
  C_forecast physical: 3506703.99 EUR
  C_forecast financial: 3506703.99 EUR
  reconciliation difference: 0.0 EUR
  positive residual (to buy): 43000.0 MWh
  negative residual (to sell): 0.0 MWh

 Coarse hedge (CAL)
  futures cost: 3477391.03 EUR
  futures profit: 55475.12 EUR
  C_forecast physical: 3451228.87 EUR
  C_forecast financial: 3451228.87 EUR
  reconciliation difference: 0.0 EUR
  positive residual (to buy): 5717.98 MWh
  negative residual (to sell): -7012.39 MWh

 Granular hedge (M01-M03, Q2-Q4)
  futures cost: 3477395.26 EUR
  futures profit: 54990.7 EUR
  C_forecast physical: 3451713.3 EUR
  C_forecast financial: 3451713.3 EUR
  reconciliation difference: 0.0 EUR
  positive residual (to buy): 5529.3 MWh
  negative residual (to sell): -6444.43 MWh
